In [ ]:

#Install Libraries
!pip install tensorflow pillow scikit-learn gradio matplotlib


In [ ]:
# Import Modules
import numpy as np
import os
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image


In [ ]:
#Folder Setup
import os
print("Images found:", os.listdir("sample_data"))


Images found: ['README.md', 'anscombe.json', 'mnist_train_small.csv', 'california_housing_train.csv', 'california_housing_test.csv', 'mnist_test.csv']


In [ ]:
import os
os.makedirs("found_outfits", exist_ok=True)


In [ ]:
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image
import numpy as np
import os

# Preprocess Dataset
# Load ResNet model (for feature extraction)
model = ResNet50(weights="imagenet", include_top=False, pooling="avg")

def extract_features(img_path):
    try:
        img = image.load_img(img_path, target_size=(224, 224))
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)
        features = model.predict(x)
        return features.flatten()
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return None

def load_and_process_images(folder):
    image_paths, features = [], []
    for file in os.listdir(folder):
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            path = os.path.join(folder, file)
            feat = extract_features(path)
            if feat is not None:
                image_paths.append(path)
                features.append(feat)
    print(f"✅ Total images processed: {len(image_paths)}")
    return image_paths, np.array(features)

image_paths, features = load_and_process_images("sample_data")

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 240ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 238ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 223ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 358ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 388ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 373ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 374ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 226ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 229ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [ ]:
 # Recommend Outfit
def recommend_outfit(uploaded_path, dataset_paths, dataset_features):
    query_features = extract_features(uploaded_path)
    sims = cosine_similarity([query_features], dataset_features)[0]
    top_indices = sims.argsort()[-5:][::-1]
    results = [(dataset_paths[i], sims[i]) for i in top_indices]
    return results

# Test using one image from your dataset
test_image = image_paths[0]
results = recommend_outfit(test_image, image_paths, features)

print("Top similar images:")
for r in results:
    print(r)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 230ms/step
Top similar images:
('sample_data/redshirt.jpg', np.float32(1.0000001))
('sample_data/marrontop.jpg', np.float32(0.8008816))
('sample_data/grayshirt.jpg', np.float32(0.7713735))
('sample_data/whiteshirt.jpg', np.float32(0.76628244))
('sample_data/redtops.jpg', np.float32(0.75814027))


In [16]:
# Similarity Graph
import matplotlib.pyplot as plt
import gradio as gr
import numpy as np

# Function to visualize graph for similarities
def plot_similarity_chart(results):
    files = [os.path.basename(f[0]) for f in results]
    scores = [f[1] for f in results]
    plt.figure(figsize=(6, 3))
    plt.barh(files, scores, color='skyblue')
    plt.xlabel("Similarity Score")
    plt.title("Top Matching Outfits")
    plt.tight_layout()

    # Save chart to temporary file
    plt.savefig("similarity_chart.png")
    plt.close()
    return "similarity_chart.png"

# Main Gradio function
def outfit_recommender(uploaded_image):
    if uploaded_image is None:
        return [], "No image uploaded."

    # Save uploaded image
    uploaded_image.save("found_outfits/temp.jpg")

    # Run recommendation
    results = recommend_outfit("found_outfits/temp.jpg", image_paths, features)

    # Prepare output
    output_images = [r[0] for r in results]
    chart = plot_similarity_chart(results)
    return output_images, chart

# Build Gradio Interface
with gr.Blocks() as app:
    gr.Markdown("## 👗 StyleMate – AI Fashion Outfit Recommender")
    gr.Markdown("Upload an outfit image to get visually similar outfit suggestions.")

    with gr.Row():
        input_image = gr.Image(label="Upload Outfit Image", type="pil")

    with gr.Row():
        output_gallery = gr.Gallery(label="Top Matching Outfits")
        output_chart = gr.Image(label="Similarity Chart")

    submit_btn = gr.Button("Submit")
    submit_btn.click(outfit_recommender, inputs=[input_image], outputs=[output_gallery, output_chart])

app.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://112abce6800a109e93.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://112abce6800a109e93.gradio.live
